# Verificacion de la calidad de la tabla plata.sucursales

Proposito del script:  
- Verificar columna por columna la calidad de los datos.

# Estableciendo Conexion

In [1]:
# Importando librerias y estableciendo conexion 
import pandas as pd 
from datetime import date
from conexiones_y_rutas import obtener_engine, obtener_ruta_archivo
engine = obtener_engine()
df_sucursales = pd.read_sql(
    "SELECT * FROM plata.sucursales",
    con=engine
)

df_sucursales_tra = df_sucursales.copy()

# Archivos de Ayuda

**Nota**: Este archivo se va utilizar para verificar las siguientes columnas:  
- [Ciudad](#ciudad)
- [Departamento](#departamento)
- [Region](#region)

In [2]:
# Importa ciudades, departamentos y regiones del peru 
df_ciuda = pd.read_csv(obtener_ruta_archivo("archivos_de_ayuda","peru_ciudades.csv"))
df_ciudades = df_ciuda.copy()

# Resumen de las Columnas

- **sucursal_id**: Identificador unico de cada sucursal.  
- **codigo_sucursal**: Codigo unico de cada sucursal SUC00(sucursal_id) => 7 digitos maximo.  
- **nombre_sucursal**: Nombre de la sucursal.  
- **tipo_sucursal**: Tipo de sucursal (ejem: Oficina Principal y Agencia).  
- **ciudad**: Ciudad donde se ubica dicha sucursal (ejem: Miraflores y Santiago de Surco).  
- **departamento**: Departamento donde se ubica dicha sucursal (ejem: Lima e Ica).  
- **region**: Region donde se ubica dicha sucursal (ejem: Lima y Callao).  
- **zona**: Zona donde se ubica dicha sucursal (ejem: Urbano y Rural).  
- **fecha_apertura**: Fecha de apertura de la sucursal.
- **estado_sucursal**: Estado actual de la sucursal (ejem: Activo e Inactivo).

# Verificacion de la Calidad de Datos

In [3]:
df_sucursales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sucursal_id      24 non-null     int64         
 1   codigo_sucursal  24 non-null     object        
 2   nombre_sucursal  24 non-null     object        
 3   tipo_sucursal    24 non-null     object        
 4   ciudad           24 non-null     object        
 5   departamento     24 non-null     object        
 6   region           24 non-null     object        
 7   zona             24 non-null     object        
 8   fecha_apertura   24 non-null     object        
 9   estado_sucursal  24 non-null     object        
 10  dwh_fecha_carga  24 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 2.2+ KB


In [4]:
df_sucursales_tra["fecha_apertura"] = pd.to_datetime(df_sucursales_tra.fecha_apertura)

In [5]:
df_sucursales_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa,2026-08-03 18:46:03.920
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa,2026-08-03 18:46:03.920
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa,2026-08-03 18:46:03.920
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2010-02-19,Activa,2026-08-03 18:46:03.920
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa,2026-08-03 18:46:03.920


In [6]:
# Verifica si existen registro duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


## sucursal_id

In [7]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.sucursal_id <= 0]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


In [8]:
# Muestra identificadores duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.sucursal_id.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


## codigo_sucursal

In [9]:
# Verifica si los formatos son correctos 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra.codigo_sucursal[
    df_sucursales_tra.codigo_sucursal 
        != df_sucursales_tra.codigo_sucursal.str.strip().str.upper()
]

Series([], Name: codigo_sucursal, dtype: object)

In [10]:
# Verifica si existe duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


In [11]:
# Verifica la extension del codigo 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal.str.len() != 7]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


In [12]:
# Recrea los codigos de sucursal, utilizando la logica observada
cod_sucur = (
    "SUC"
    + df_sucursales_tra["sucursal_id"]
        .astype(str)
        .str.zfill(4)
)
# Verifica si los codigos generados son iguales a los codigos existentes
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal != cod_sucur]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


## nombre_sucursal

**Nota**: Tengo dudas sobre el nombre, porque, parecer ser una union de varias columnas:  
- Si es oficina principal es tipo_sucursal + departamento.  
- Si es agencia es tipo_sucursal + ciudad.  
Pero tambien existen "oficinas especiales" y "puntos de atencion" y estos valores no siguen esta regla, asi que no estoy muy seguro, como no puedo resolver esta duda voy a dejar los nombres como estan.

In [13]:
# Verifica el formato de los nombres
# Resultados Esperados: Agencia Santiago de Surco 1, Agencia los Olivos 3, Oficina Principal la Libertad, Agencia el Tambo 1
df_sucursales_tra.nombre_sucursal[df_sucursales_tra.nombre_sucursal 
                                != df_sucursales_tra.nombre_sucursal.str.strip().str.title()]

1       Agencia Santiago de Surco 1
3              Agencia los Olivos 3
8     Oficina Principal la Libertad
18               Agencia el Tambo 1
Name: nombre_sucursal, dtype: object

## tipo_sucursal

In [14]:
# Resultados Esperados: 'Oficina Principal', 'Agencia', 'Punto De Atención', 'Oficina Especial'
df_sucursales_tra.tipo_sucursal.unique()

array(['Oficina Principal', 'Agencia', 'Punto de Atención',
       'Oficina Especial'], dtype=object)

In [15]:
# Resultados Esperados: Punto de Atención
df_sucursales_tra.tipo_sucursal[df_sucursales_tra.tipo_sucursal 
                                != df_sucursales_tra.tipo_sucursal.str.strip().str.title()]

2     Punto de Atención
6     Punto de Atención
14    Punto de Atención
18    Punto de Atención
Name: tipo_sucursal, dtype: object

## ciudad

In [16]:
# Verifica si las ciudades siguen el formato adecuado 
# Resultados Esperados: San Juan de Lurigancho, Santiago de Surco
df_sucursales_tra.ciudad[df_sucursales_tra.ciudad 
                        != df_sucursales_tra.ciudad.str.strip().str.title()]

0    San Juan de Lurigancho
1         Santiago de Surco
Name: ciudad, dtype: object

In [17]:
# Valida si todas las ciudades de sucursales, se encuentra en el registro de la empresa 
# Resultados esperados: both: 24, left_only: 0, right_only: 0
verficar_ciudad = df_sucursales_tra.merge(
    right=df_ciudades,
    on="ciudad",
    indicator=True,
    suffixes=["_sucur_tra","_ciudades"]
)
verficar_ciudad._merge.value_counts()

_merge
both          24
left_only      0
right_only     0
Name: count, dtype: int64

## departamento

In [18]:
# Verifica si los departamentos siguen el formato adecuado 
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.departamento[df_sucursales_tra.departamento 
                            != df_sucursales_tra.departamento.str.strip().str.title()]

Series([], Name: departamento, dtype: object)

In [19]:
# Valida si para la ciudad el departamento es el correcto 
# Resultados esperados: Tabla Vacía
verificar_departamento = df_sucursales_tra.merge(
    right=df_ciudades,
    on="ciudad",
    suffixes=["_sucur_tra","_ciudades"]
)
verificar_departamento[verificar_departamento.departamento_sucur_tra != verificar_departamento.departamento_ciudades]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento_sucur_tra,region_sucur_tra,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga,departamento_ciudades,region_ciudades


## region

In [20]:
# Valida si la region tiene el formato correcto
# Resultados esperados: Lima y Callao
df_sucursales_tra.region[df_sucursales_tra.region 
                        != df_sucursales_tra.region.str.strip().str.title()]

0    Lima y Callao
1    Lima y Callao
2    Lima y Callao
3    Lima y Callao
4    Lima y Callao
Name: region, dtype: object

In [21]:
# Valida si para la ciudad la region es la correcta 
# Resultados esperados: Tabla Vacia
verificar_region = df_sucursales_tra.merge(right=df_ciudades,
                                        on="ciudad",
                                        suffixes=["_sucur_tra","_ciudades"]
                                    )
verificar_region[verificar_region.region_sucur_tra != verificar_region.region_ciudades]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento_sucur_tra,region_sucur_tra,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga,departamento_ciudades,region_ciudades


## zona

In [22]:
# Resultados Esperados: 'Urbano', 'Selva','Rural', 'n/a'
df_sucursales_tra.zona.unique()

array(['Urbano', 'Selva'], dtype=object)

## fecha_apertura

In [23]:
# Verifica si existen fechas que generen errores 
# Resultado Esperado: Tabla Vacia
fechas_apertura_error = pd.to_datetime(
    df_sucursales_tra.fecha_apertura,
    errors='coerce'
)
df_sucursales_tra.fecha_apertura[fechas_apertura_error.isna()]

Series([], Name: fecha_apertura, dtype: datetime64[ns])

In [ ]:
# Verifica que las fechas de apertura no sean futuras 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.fecha_apertura.dt.date >= date.today()]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal,dwh_fecha_carga


## estado_sucursal

In [25]:
# Resultados Esperados: 'Activa', 'Inactiva', 'n/a'
df_sucursales_tra.estado_sucursal.unique()

array(['Activa'], dtype=object)